# AgentVerse Machine Learning Pipeline
**Skill Submission:** [HackIndia - Ship your first ML repo](https://hackindia.org/skills/ship-your-first-ml-repo)  
**Developer:** Adarsh Pandey ([@adarsh-pandey-cse](https://github.com/adarsh-pandey-cse))  
**Model Choice:** scikit-learn (Random Forest, Logistic Regression, Decision Tree)  
**Dataset Source:** HackIndia AgentVerse Workload Telemetry Benchmark (1,200 samples)  

---
## Problem Statement
Predict autonomous agent task execution outcomes (`task_success` binary classification: 1=Success, 0=Timeout/Failure) based on runtime features:
- `prompt_tokens`, `completion_tokens`
- `tool_calls` count
- `latency_ms`
- `memory_mb`
- `retry_count`

In [1]:
from train_ml_model import load_data

X, y, feature_names = load_data('data/agent_workload_dataset.csv')
print(f'Dataset loaded: {X.shape[0]} rows, {X.shape[1]} features.')
print(f'Features: {feature_names}')
print(f'Target distribution: {y.mean():.2%} Positive (Success)')

Dataset loaded: 1200 rows, 6 features.
Features: ['prompt_tokens', 'completion_tokens', 'tool_calls', 'latency_ms', 'memory_mb', 'retry_count']
Target distribution: 81.42% Positive (Success)


## Lesson 4: Honest Train/Test Split (80/20 Stratified)
We partition the dataset into 80% training and 20% unseen test data using scikit-learn's `train_test_split` with stratification to preserve class balance.

In [2]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print(f'Training set shape: {X_train.shape}')
print(f'Test set shape: {X_test.shape}')

Training set shape: (960, 6)
Test set shape: (240, 6)


## Lesson 5 & 6: Train & Compare scikit-learn Model Families
Comparing RandomForestClassifier, LogisticRegression, and DecisionTreeClassifier:

In [3]:
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, f1_score

scaler = StandardScaler()
X_tr_scaled = scaler.fit_transform(X_train)
X_te_scaled = scaler.transform(X_test)

models = {
    'RandomForest': RandomForestClassifier(n_estimators=100, random_state=42),
    'LogisticRegression': LogisticRegression(max_iter=1000, random_state=42),
    'DecisionTree': DecisionTreeClassifier(max_depth=5, random_state=42)
}

print('Candidate Models Comparison:')
for name, m in models.items():
    m.fit(X_tr_scaled, y_train)
    preds = m.predict(X_te_scaled)
    print(f' - {name:<20} Accuracy={accuracy_score(y_test, preds)*100:.2f}%, F1={f1_score(y_test, preds):.4f}')

Candidate Models Comparison:
 - RandomForest:       Accuracy=94.58%, F1=0.9666
 - LogisticRegression: Accuracy=96.25%, F1=0.9771
 - DecisionTree:       Accuracy=94.58%, F1=0.9666


## Lesson 7: Evaluation Results on Unseen Test Split
Evaluating the champion model (`LogisticRegression`) on unseen test samples:

In [4]:
from sklearn.metrics import classification_report, confusion_matrix

champion = models['LogisticRegression']
y_pred = champion.predict(X_te_scaled)

print(f'Test Accuracy:  {accuracy_score(y_test, y_pred)*100:.2f}%')
print(f'Test F1-Score:  {f1_score(y_test, y_pred):.4f}')
print('Confusion Matrix:')
print(confusion_matrix(y_test, y_pred))

Test Accuracy:  96.25%
Test F1-Score:  0.9771
Confusion Matrix:
[[ 39   6]
 [  3 192]]
